In [ ]:
# 修改预训练模型的缓存目录
import os

os.environ["MODELSCOPE_CACHE"] = r"G:\code\pretrain_model_dir\_modelscope"

In [ ]:
from transformers import AutoModel
from transformers.utils.versions import require_version


require_version(
    "transformers<4.52.0",
    "The remote code has some issues with transformers>=4.52.0, please downgrade: pip install transformers==4.51.3"
)

# TODO: 这样加载不行
gme = AutoModel.from_pretrained(
    # "Alibaba-NLP/gme-Qwen2-VL-2B-Instruct",
    r"G:\code\pretrain_model_dir\_modelscope\gme-Qwen2-VL-2B-Instruct",
    torch_dtype="float16", device_map='cuda', trust_remote_code=True
)


In [ ]:
import sys
sys.path.append(r"G:\code\pretrain_model_dir\_modelscope\gme-Qwen2-VL-2B-Instruct")

# You can find the script gme_inference.py in https://modelscope.cn/models/iic/gme-Qwen2-VL-7B-Instruct/file/view/master?fileName=gme_inference.py
from gme_inference import GmeQwen2VL

texts = [
    "What kind of car is this?",
    "The Tesla Cybertruck is a battery electric pickup truck built by Tesla, Inc. since 2023."
]
images = [
    'https://mitalinlp.oss-cn-hangzhou.aliyuncs.com/test/Tesla_Cybertruck_damaged_window.jpg',
    'https://mitalinlp.oss-cn-hangzhou.aliyuncs.com/test/2024_Tesla_Cybertruck_Foundation_Series%2C_front_left_(Greenwich).jpg',
]

gme = GmeQwen2VL(r"G:\code\pretrain_model_dir\_modelscope\gme-Qwen2-VL-2B-Instruct")
gme

In [ ]:
# Single-modal embedding
e_text = gme.get_text_embeddings(texts=texts)
e_image = gme.get_image_embeddings(images=images)
print((e_text * e_image).sum(-1))
## tensor([0.2281, 0.6001], dtype=torch.float16)

In [ ]:
# Single-modal embedding
e_text = gme.get_text_embeddings(texts=texts)
e_image = gme.get_image_embeddings(images=images)
print('Single-modal', (e_text @ e_image.T).tolist())
## Single-modal [[0.359619140625, 0.0655517578125], [0.04180908203125, 0.374755859375]]

In [ ]:
# How to set embedding instruction
t2i_prompt = 'Find an image that matches the given text.'
e_query = gme.get_text_embeddings(texts=texts, instruction=t2i_prompt)
# If is_query=False, we always use the default instruction.
e_corpus = gme.get_image_embeddings(images=images, is_query=False)
print('Single-modal with instruction', (e_query @ e_corpus.T).tolist())
## Single-modal with instruction [[0.429931640625, 0.11505126953125], [0.049835205078125, 0.409423828125]]


In [ ]:
# Fused-modal embedding
e_fused = gme.get_fused_embeddings(texts=texts, images=images)
print('Fused-modal', (e_fused @ e_fused.T).tolist())
## Fused-modal [[1.0, 0.05511474609375], [0.05511474609375, 1.0]]